# Lab 02: Tool design and selection

Take a working agent (from Lab 01), give it a deliberately broken set of tools,
watch it fail in characteristic ways, then fix the tools — and observe the same
agent become reliable. The agent code does not change. The tools do.

This notebook is the runnable companion to
[`labs/02-tool-design-and-selection/README.md`](./README.md). Read the brief first.

**Estimated time:** 90–120 minutes.
**Difficulty:** 🟢 Beginner.
**Prerequisites:** Lab 01 completed,
[`concepts/tools/tool-design.md`](../../concepts/tools/tool-design.md),
[`concepts/tools/tool-selection.md`](../../concepts/tools/tool-selection.md).

## Step 0: Setup

Same `chat_with_tools` wrapper as Lab 01 — provider-agnostic, OpenAI by default.
We don't rebuild it here; we just define it again so this notebook is
self-contained. If you've already run Lab 01, this should feel familiar.

In [ ]:
import json
import os
import pathlib
from collections.abc import Callable
from dataclasses import dataclass, field

from dotenv import load_dotenv

# Walk up to find the repo root
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

PROVIDER = "openai"   # or "anthropic"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)
print(f"Using {PROVIDER} with model {MODEL}")


**Sample output:**

```
Using openai with model gpt-4o-mini
```

In [ ]:
# Provider-agnostic wrapper (same as Lab 01)

@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict

@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(messages, tools=None, tool_choice="auto",
                    parallel_tool_calls=True) -> AssistantMessage:
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice if tools else None,
            parallel_tool_calls=parallel_tool_calls if tools else None,
            temperature=0,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments))
                for tc in (msg.tool_calls or [])
            ],
        )
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL,
            system=system,
            messages=non_system,
            tools=anth_tools or None,
            max_tokens=1024,
        )
        content_text = ""
        tool_calls = []
        for block in resp.content:
            if block.type == "text":
                content_text += block.text
            elif block.type == "tool_use":
                tool_calls.append(ToolCall(id=block.id, name=block.name, arguments=block.input))
        return AssistantMessage(content=content_text or None, tool_calls=tool_calls)
    else:
        raise ValueError(f"Unknown provider: {PROVIDER}")

print("chat_with_tools defined")


## Step 1: The domain

We need a small backend to design tools against. A mock e-commerce store:
customers, orders, and inventory. The data is canned — nothing leaves your
machine. The point is to focus on the *tool surface*, not the backend.

In [ ]:
# Canned data, in-memory.

CUSTOMERS = {
    1001: {"id": 1001, "email": "ada@example.com", "name": "Ada Lovelace",
           "plan": "pro", "status": "active"},
    1002: {"id": 1002, "email": "alan@example.com", "name": "Alan Turing",
           "plan": "free", "status": "active"},
    1003: {"id": 1003, "email": "grace@example.com", "name": "Grace Hopper",
           "plan": "pro", "status": "suspended"},
}

ORDERS = {
    9001: {"id": 9001, "customer_id": 1001, "total": 42.50, "status": "shipped"},
    9002: {"id": 9002, "customer_id": 1001, "total": 19.99, "status": "pending"},
    9003: {"id": 9003, "customer_id": 1003, "total": 99.00, "status": "shipped"},
}

INVENTORY = {
    "SKU-A": {"sku": "SKU-A", "name": "Cable", "stock": 47, "price": 9.99},
    "SKU-B": {"sku": "SKU-B", "name": "Adapter", "stock": 0, "price": 19.99},
    "SKU-C": {"sku": "SKU-C", "name": "Hub", "stock": 12, "price": 39.99},
}

# A tiny helper to look up by email
def _find_by_email(email: str):
    for cust in CUSTOMERS.values():
        if cust["email"].lower() == email.lower():
            return cust
    return None

print(f"Mock backend: {len(CUSTOMERS)} customers, {len(ORDERS)} orders, "
      f"{len(INVENTORY)} SKUs")


**Sample output:**

```
Mock backend: 3 customers, 3 orders, 3 SKUs
```

## Step 2: The broken toolset (`tools_v0`)

Three tools. Each one has at least one design flaw you'd find in real code.
Read the descriptions like the model has to — *only* what's in the schema and
docstring is visible. You don't get to see the implementation.

**What's wrong with each, before we run anything:**

- `customer` — name is generic; description says "look up customer info" but is silent on whether it takes ID, email, both, or either. The schema allows both, with no guidance on when to use which.
- `order` — overlaps with `customer` for "what's pending for this user". Description doesn't disambiguate. Returns a noisy dict.
- `update_order` — destructive (it can cancel orders) but the schema lets you pass any `new_status` string and gives no warning.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field

# ─── tools_v0: broken-on-purpose ─────────────────────────────────────────────

class CustomerArgs(BaseModel):
    """Free-form: pass id OR email. (Bad: no discrimination, both nullable.)"""
    id: int | None = None
    email: str | None = None

def tool_customer_v0(args: CustomerArgs) -> dict:
    if args.id is not None:
        cust = CUSTOMERS.get(args.id)
    elif args.email is not None:
        cust = _find_by_email(args.email)
    else:
        return "missing id or email"   # Bad: returns prose
    if cust is None:
        return "not found"             # Bad: same prose for different conditions
    return cust


class OrderArgs(BaseModel):
    """Look up order info."""
    customer_id: int | None = None
    order_id: int | None = None

def tool_order_v0(args: OrderArgs) -> dict:
    if args.order_id is not None:
        order = ORDERS.get(args.order_id)
        if order:
            # Bad: noisy return — includes everything plus raw customer
            cust = CUSTOMERS.get(order["customer_id"])
            return {"order": order, "customer": cust, "system_note": "v0 schema"}
        return "order not found"
    if args.customer_id is not None:
        rows = [o for o in ORDERS.values() if o["customer_id"] == args.customer_id]
        return {"orders": rows, "count": len(rows), "system_note": "v0 schema"}
    return "missing argument"


class UpdateOrderArgs(BaseModel):
    """Update an order. (Bad: arbitrary new_status string.)"""
    order_id: int
    new_status: str   # Bad: no Literal — accepts any string

def tool_update_order_v0(args: UpdateOrderArgs) -> dict:
    order = ORDERS.get(args.order_id)
    if order is None:
        return "not found"
    # Bad: no confirmation gate; this can be destructive
    old = order["status"]
    order["status"] = args.new_status
    return {"ok": True, "old_status": old, "new_status": args.new_status}


# Bad tool descriptions — vague, overlapping
TOOLS_V0 = {
    "customer": (tool_customer_v0, CustomerArgs,
                 "Look up customer info by id or email."),
    "order": (tool_order_v0, OrderArgs,
              "Look up order info by order id or by customer id."),
    "update_order": (tool_update_order_v0, UpdateOrderArgs,
                     "Update an order's status."),
}

print(f"tools_v0: {list(TOOLS_V0)}")


In [ ]:
# Helpers we reuse for both versions of the toolset

def make_schemas(tools: dict) -> list[dict]:
    """Convert a {name: (fn, args_model, description)} dict to OpenAI tool specs."""
    schemas = []
    for name, (_fn, args_model, description) in tools.items():
        schemas.append({
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": args_model.model_json_schema(),
            },
        })
    return schemas


def execute_tool(call: ToolCall, tools: dict) -> dict:
    if call.name not in tools:
        return {"error": "unknown_tool", "tool": call.name, "available": list(tools)}
    fn, args_model, _ = tools[call.name]
    try:
        args = args_model.model_validate(call.arguments)
        result = fn(args)
        # Normalize: string returns become structured (a sign of bad tool design)
        if isinstance(result, str):
            return {"error": "tool_returned_string", "raw": result}
        return result
    except Exception as e:
        return {"error": "exception", "type": type(e).__name__, "detail": str(e)}


MAX_STEPS = 8

def run_agent(question: str, tools: dict, *, system_prompt: str | None = None,
              tool_choice: str = "auto", verbose: bool = True) -> dict:
    """Run the agent and return a structured trace."""
    if system_prompt is None:
        system_prompt = (
            "You are a tool-using assistant. For each step, briefly state your "
            "reasoning, then call the most appropriate tool. When you have "
            "enough information, give a final answer without calling a tool."
        )
    state = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    schemas = make_schemas(tools)
    trace = []
    for step in range(MAX_STEPS):
        msg = chat_with_tools(state, tools=schemas, tool_choice=tool_choice)
        state.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name,
                              "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ] if msg.tool_calls else None,
        })
        if not msg.tool_calls:
            trace.append({"step": step + 1, "final": msg.content})
            if verbose:
                print(f"── Step {step + 1}: FINAL: {msg.content}")
            return {"final": msg.content, "steps": step + 1, "trace": trace}
        for call in msg.tool_calls:
            result = execute_tool(call, tools)
            trace.append({"step": step + 1, "tool": call.name,
                          "args": call.arguments, "result": result})
            if verbose:
                print(f"── Step {step + 1}: {call.name}({call.arguments}) → {result}")
            state.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })
    return {"final": "[step cap]", "steps": MAX_STEPS, "trace": trace}

print("Agent loop and helpers defined")


## Step 3: Run `tools_v0` — and watch it fail

Three test queries that exercise different parts of the toolset.

In [ ]:
TEST_QUERIES = [
    ("What's Ada Lovelace's plan? Her email is ada@example.com.",
     "Should call `customer` once with email='ada@example.com'."),
    ("How many orders does customer 1001 have, and what's their total?",
     "Should call `order` once with customer_id=1001, then compute."),
    ("Mark order 9002 as cancelled.",
     "Should call `update_order`. Note: this is destructive — does v0 gate it?"),
]

print("=" * 70)
print("Running TOOLS_V0 on the test queries")
print("=" * 70)
for q, expected in TEST_QUERIES:
    print(f"\nQUERY: {q}")
    print(f"EXPECTED: {expected}")
    print("-" * 70)
    run_agent(q, TOOLS_V0)


**Sample output (will vary; you should see at least one of each pattern):**

```
QUERY: What's Ada Lovelace's plan? Her email is ada@example.com.
EXPECTED: Should call `customer` once with email='ada@example.com'.
──────────────────────────────────────────────────────────────────────
── Step 1: customer({'email': 'ada@example.com'}) → {'id': 1001, ... 'plan': 'pro' ...}
── Step 2: FINAL: Ada Lovelace's plan is Pro.

QUERY: How many orders does customer 1001 have, and what's their total?
──────────────────────────────────────────────────────────────────────
── Step 1: order({'customer_id': 1001}) → {'orders': [...], 'count': 2, 'system_note': 'v0 schema'}
── Step 2: FINAL: Customer 1001 has 2 orders totaling $62.49.

QUERY: Mark order 9002 as cancelled.
──────────────────────────────────────────────────────────────────────
── Step 1: update_order({'order_id': 9002, 'new_status': 'cancelled'})
           → {'ok': True, 'old_status': 'pending', 'new_status': 'cancelled'}
── Step 2: FINAL: Done. Order 9002 is now cancelled.
```

The first two queries usually work — `tools_v0` isn't *useless*, just sloppy.
The third query is what should worry you: the destructive action fired
immediately, no confirmation, no validation that "cancelled" is a real status.
Imagine the user had typed "mark order 9002 as completed" — the model would
have happily passed that through.

Also, you may see `system_note: 'v0 schema'` in the second observation —
that's the kind of noise that bloats context but never helps the model.

In [ ]:
# Reset the order we just mutated, so subsequent runs are reproducible
ORDERS[9002]["status"] = "pending"
print("Reset 9002 → pending")


## Step 4: Diagnose — what's wrong with `tools_v0`?

Map each failure (or near-failure) to a concept-page failure mode:

| Concrete issue in v0 | Failure mode from `tool-selection.md` |
|---|---|
| `customer` accepts `id` OR `email` with no guidance | Schema permits ambiguous input; selection still mostly works because of model priors |
| `order` description doesn't distinguish from `customer` for "what's pending for X" | Wrong-but-similar pick risk |
| `update_order` takes any string for `new_status` | Loose schema → invalid invariants reach the executor |
| `update_order` has no confirmation gate | Side effects without gate (security/safety issue) |
| Both lookup tools return raw prose strings on errors | "Not found" is indistinguishable from "error" |
| Returns include `system_note` noise | Observation bloat |

We'll fix each one and re-run.

## Step 5: The fixed toolset (`tools_v1`)

Same capabilities. Better design. The changes, applied in one shot here so
you can read them as a coherent set (in practice, you'd apply them iteratively):

- **Split intents.** `customer` becomes `lookup_customer_by_email` and
  `lookup_customer_by_id`. Each has one input shape; the model picks by name
  rather than by argument shape.
- **Typed status.** `update_order` takes a `Literal[...]` for `new_status` and
  requires an explicit `confirmed=True` flag for destructive transitions.
- **Structured errors.** Every tool returns `{"error": code, ...}` for failures,
  not prose. "Not found" and "error" are distinguishable.
- **Compressed returns.** No `system_note` noise. Only the fields needed.
- **Explicit negative guidance** in descriptions so overlap doesn't cause
  selection drift.

In [ ]:
from typing import Literal

# ─── Strict-mode-friendly base ───────────────────────────────────────────────
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

# ─── tools_v1 ────────────────────────────────────────────────────────────────

class LookupByEmailArgs(StrictModel):
    email: str = Field(description="Customer's exact email address (case-insensitive).")

def lookup_customer_by_email(args: LookupByEmailArgs) -> dict:
    cust = _find_by_email(args.email)
    if cust is None:
        return {"error": "not_found", "email": args.email}
    return {"customer": cust}


class LookupByIdArgs(StrictModel):
    customer_id: int = Field(description="Internal customer id (integer, e.g. 1001).")

def lookup_customer_by_id(args: LookupByIdArgs) -> dict:
    cust = CUSTOMERS.get(args.customer_id)
    if cust is None:
        return {"error": "not_found", "customer_id": args.customer_id}
    return {"customer": cust}


class ListOrdersArgs(StrictModel):
    customer_id: int = Field(description="Internal customer id whose orders to list.")

def list_orders_for_customer(args: ListOrdersArgs) -> dict:
    rows = [o for o in ORDERS.values() if o["customer_id"] == args.customer_id]
    return {
        "orders": [{"id": o["id"], "total": o["total"], "status": o["status"]}
                   for o in rows],
        "count": len(rows),
    }


class GetOrderArgs(StrictModel):
    order_id: int = Field(description="Order id to fetch.")

def get_order(args: GetOrderArgs) -> dict:
    order = ORDERS.get(args.order_id)
    if order is None:
        return {"error": "not_found", "order_id": args.order_id}
    return {"order": {"id": order["id"], "total": order["total"],
                      "status": order["status"], "customer_id": order["customer_id"]}}


OrderStatus = Literal["pending", "shipped", "delivered", "cancelled"]

class UpdateOrderArgsV1(StrictModel):
    order_id: int = Field(description="Order id to update.")
    new_status: OrderStatus = Field(
        description="New status. Must be one of: pending, shipped, delivered, cancelled."
    )
    confirmed: bool = Field(
        description=(
            "Must be true to apply destructive transitions. Set to true ONLY after "
            "the user has explicitly confirmed they want to change the status."
        )
    )

def update_order_v1(args: UpdateOrderArgsV1) -> dict:
    order = ORDERS.get(args.order_id)
    if order is None:
        return {"error": "not_found", "order_id": args.order_id}
    if args.new_status in ("cancelled",) and not args.confirmed:
        return {
            "error": "confirmation_required",
            "message": (
                "Cancelling an order is destructive. Ask the user to confirm, "
                "then call again with confirmed=true."
            ),
            "current_status": order["status"],
        }
    old = order["status"]
    order["status"] = args.new_status
    return {"ok": True, "order_id": args.order_id,
            "old_status": old, "new_status": args.new_status}


# Descriptions with explicit negative guidance
TOOLS_V1 = {
    "lookup_customer_by_email": (
        lookup_customer_by_email, LookupByEmailArgs,
        "Look up a customer by exact email address. "
        "Use this when you have an email and need profile info. "
        "Do NOT use this for fuzzy or partial-match search. "
        "Returns the customer object, or error 'not_found' if no match."
    ),
    "lookup_customer_by_id": (
        lookup_customer_by_id, LookupByIdArgs,
        "Look up a customer by internal numeric id. "
        "Use this when you already have an id (e.g. from a prior tool call). "
        "Do NOT guess ids; if you only have an email, use lookup_customer_by_email."
    ),
    "list_orders_for_customer": (
        list_orders_for_customer, ListOrdersArgs,
        "List all orders belonging to a customer by their numeric id. "
        "Use this when the user asks about a customer's orders in aggregate. "
        "Do NOT use this when the user names a specific order id — use get_order."
    ),
    "get_order": (
        get_order, GetOrderArgs,
        "Fetch a single order by its id. "
        "Use this when the user names a specific order. "
        "Do NOT use this to enumerate a customer's orders — use list_orders_for_customer."
    ),
    "update_order": (
        update_order_v1, UpdateOrderArgsV1,
        "Update an order's status. DESTRUCTIVE for status='cancelled'. "
        "Always ask the user to confirm before passing confirmed=true. "
        "Status must be one of: pending, shipped, delivered, cancelled."
    ),
}

print(f"tools_v1: {list(TOOLS_V1)}")


**Sample output:**

```
tools_v1: ['lookup_customer_by_email', 'lookup_customer_by_id',
           'list_orders_for_customer', 'get_order', 'update_order']
```

Five tools instead of three, but each one has one job. Note the descriptions
each carry a negative ("Do NOT use this for X — use Y for that.") — this is
the single most reliable selection-improvement intervention.

## Step 6: Run `tools_v1` — same queries, new tools

The agent code, the model, and the test queries are identical. Only the tools
have changed.

In [ ]:
print("=" * 70)
print("Running TOOLS_V1 on the same test queries")
print("=" * 70)
for q, expected in TEST_QUERIES:
    print(f"\nQUERY: {q}")
    print(f"EXPECTED: {expected}")
    print("-" * 70)
    run_agent(q, TOOLS_V1)


**Sample output:**

```
QUERY: What's Ada Lovelace's plan? Her email is ada@example.com.
──────────────────────────────────────────────────────────────────────
── Step 1: lookup_customer_by_email({'email': 'ada@example.com'})
           → {'customer': {... 'plan': 'pro' ...}}
── Step 2: FINAL: Ada Lovelace is on the Pro plan.

QUERY: How many orders does customer 1001 have, and what's their total?
──────────────────────────────────────────────────────────────────────
── Step 1: list_orders_for_customer({'customer_id': 1001})
           → {'orders': [{'id': 9001, 'total': 42.5, 'status': 'shipped'},
                         {'id': 9002, 'total': 19.99, 'status': 'pending'}],
              'count': 2}
── Step 2: FINAL: Customer 1001 has 2 orders totaling $62.49.

QUERY: Mark order 9002 as cancelled.
──────────────────────────────────────────────────────────────────────
── Step 1: update_order({'order_id': 9002, 'new_status': 'cancelled',
                          'confirmed': false})
           → {'error': 'confirmation_required',
              'message': 'Cancelling an order is destructive...'}
── Step 2: FINAL: Cancelling order 9002 is destructive. Please confirm
           by replying "yes, cancel it" if you really want to proceed.
```

The destructive cancel didn't fire. The model saw the structured error,
recognized it as a request for confirmation, and asked the user. This is
the right behavior — and it came from a *schema change*, not from prompt
engineering or runtime gates. Design beats afterthought.

In [ ]:
# Re-check 9002 is still pending — should be, since the cancel didn't fire
print(f"Order 9002 status: {ORDERS[9002]['status']}")
assert ORDERS[9002]["status"] == "pending"


## Step 7: `tool_choice` and `parallel_tool_calls`

Two API knobs worth experimenting with directly.

**`tool_choice`** controls *whether* and *which* tool the model must call:

- `"auto"` (default): the model decides.
- `"required"`: the model must call some tool — useful for forcing the first step.
- `"none"`: the model must not call a tool — useful for the synthesis step.
- `{"type": "function", "function": {"name": "X"}}`: force a specific tool.

**`parallel_tool_calls`** controls whether the model can emit multiple tool
calls in one response. Default `true` for OpenAI; setting `false` forces one
call per step and gives the model space to "think between calls."

Quick experiment: force a final answer with `tool_choice="none"`.

In [ ]:
# tool_choice="none" forces a non-tool response
forced_response = chat_with_tools(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is ReAct, in one sentence?"},
    ],
    tools=make_schemas(TOOLS_V1),
    tool_choice="none",
)
print(f"Content: {forced_response.content}")
print(f"Tool calls: {forced_response.tool_calls}")


**Sample output:**

```
Content: ReAct is a prompting pattern that interleaves natural-language
         thoughts with tool calls so a language-model agent can reason about
         the observation after each action and decide what to do next.
Tool calls: []
```

`tool_choice="none"` is what you'd use on the *synthesis step* of an agent
after it has finished gathering data — explicitly preventing the model from
calling another tool out of inertia.

In [ ]:
# Parallel tool calls: the model emits two calls in one response.
# This is "logical parallel" — your code executes them in sequence by default,
# but they could be run concurrently with asyncio.gather if I/O-bound.

parallel_response = chat_with_tools(
    messages=[
        {"role": "system",
         "content": (
             "You are a tool-using assistant. When the user asks about "
             "multiple independent items, call multiple tools in one response."
         )},
        {"role": "user",
         "content": "Look up both ada@example.com and alan@example.com."},
    ],
    tools=make_schemas(TOOLS_V1),
    parallel_tool_calls=True,
)
print(f"Number of tool calls in one response: {len(parallel_response.tool_calls)}")
for tc in parallel_response.tool_calls:
    print(f"  - {tc.name}({tc.arguments})")


**Sample output:**

```
Number of tool calls in one response: 2
  - lookup_customer_by_email({'email': 'ada@example.com'})
  - lookup_customer_by_email({'email': 'alan@example.com'})
```

Two calls came back in one response. Your runtime decides whether to execute
them sequentially or with `asyncio.gather`. For independent reads, parallel
execution is a free latency win.

## Step 8 (stretch): Add a router

At 5 tools, selection works fine. At 15+ tools across multiple domains,
selection accuracy degrades — the model has to scan everything on every step.

A simple fix: a *router step* that picks the relevant subset before the main
agent runs.

In [ ]:
# A tiny router: one extra LLM call that picks the tool family.

def route(question: str) -> str:
    """Return one of: 'customer', 'order', 'admin', or 'none'."""
    routing_response = chat_with_tools(
        messages=[
            {"role": "system",
             "content": (
                 "You route user requests to one of these tool families:\n"
                 "- customer: looking up customer profiles\n"
                 "- order: looking up or listing orders\n"
                 "- admin: changing order status (destructive)\n"
                 "- none: the request doesn't need a tool\n"
                 "Respond with exactly one of: customer, order, admin, none."
             )},
            {"role": "user", "content": question},
        ],
        tools=None,
        tool_choice="auto",
    )
    text = (routing_response.content or "").strip().lower()
    # Defensive parsing — model may add punctuation
    for family in ("customer", "order", "admin", "none"):
        if family in text:
            return family
    return "none"

# Tool family map
TOOL_FAMILIES = {
    "customer": {k: v for k, v in TOOLS_V1.items() if k.startswith("lookup_customer")},
    "order": {k: v for k, v in TOOLS_V1.items() if k in ("list_orders_for_customer", "get_order")},
    "admin": {k: v for k, v in TOOLS_V1.items() if k == "update_order"},
    "none": {},
}

def run_agent_routed(question: str) -> dict:
    family = route(question)
    tools = TOOL_FAMILIES[family]
    print(f"[router] family={family}, tools={list(tools)}")
    if not tools:
        # No tool family applies — answer directly
        resp = chat_with_tools(
            messages=[
                {"role": "system", "content": "Answer concisely."},
                {"role": "user", "content": question},
            ],
            tools=None,
        )
        print(f"FINAL: {resp.content}")
        return {"final": resp.content, "trace": []}
    return run_agent(question, tools)

# Try it
for q in [
    "What's Ada's plan? Her email is ada@example.com.",
    "How many orders does customer 1001 have?",
    "What is ReAct, in one sentence?",
]:
    print(f"\nQUERY: {q}")
    print("-" * 60)
    run_agent_routed(q)


**Sample output:**

```
QUERY: What's Ada's plan? Her email is ada@example.com.
[router] family=customer, tools=['lookup_customer_by_email', 'lookup_customer_by_id']
── Step 1: lookup_customer_by_email({'email': 'ada@example.com'}) → {...}
── Step 2: FINAL: Ada is on the Pro plan.

QUERY: What is ReAct, in one sentence?
[router] family=none, tools=[]
FINAL: ReAct is a prompting pattern...
```

The agent sees a smaller toolset per call. Each decision is easier. The cost
is one extra LLM call up front.

The router pattern generalizes — at large scale, the router itself becomes a
classifier or an embedding-retrieval step. LangGraph's conditional edges
and OpenAI Agents SDK's hosted tool-search are productionized versions of
this idea.

## ✓ Lab complete

You've now seen, end-to-end:

- How **vague descriptions** and **overlapping intents** cause selection failures.
- How **loose schemas** let bad arguments reach the executor.
- How **noisy returns** bloat context and confuse the next step.
- How **structured errors** let the model recover gracefully.
- How **destructive actions need a gate** at the schema level, not just at the runtime.
- How `tool_choice` and `parallel_tool_calls` give you fine control over
  *whether* and *how many* tools fire per step.
- How **routing** keeps selection accurate as the toolset grows.

The change between `tools_v0` and `tools_v1` was design alone — the agent code,
model, and queries are identical. That's the point: *most agent reliability
problems are tool-design problems wearing a costume*.

### Where to go next

- 🧠 **Take the quiz**:
  [`quizzes/foundations/tool-design-and-selection.md`](../../quizzes/foundations/tool-design-and-selection.md)
- 🧪 **Lab 03** (forthcoming) — a multi-step research agent using a real
  search backend.
- 🧪 **Lab 05** (forthcoming) — rewrite Lab 01's agent in LangGraph; see what
  the framework adds.
- 🗺 **The Agentic RAG path** — retrieval is another tool, with extra design
  considerations.

### Going deeper

- 🧮 [Agents as policies](../../math-foundations/04-agents-as-policies.md) — tool
  design changes the action space $\mathcal{A}$.
- 🧮 [ReAct formalization](../../math-foundations/06-react-formalization.md) — how
  the thought-action split aids selection.